<a href="https://colab.research.google.com/github/AdityaaaTiwari/AdityaaaTiwari/blob/main/NYC_Taxi_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [28]:
jan_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
feb_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet"
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

trip_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type"
]

jan = pd.read_parquet(jan_url, columns=trip_cols)

feb = pd.read_parquet(feb_url, columns=trip_cols)

zones = pd.read_csv(zone_url)

trips = pd.concat(
    [jan, feb],
    ignore_index=True
)

print(trips.shape)

(5972150, 9)


In [24]:
print("First 5 Rows")
print(trips.head())

print("\nShape")
print(trips.shape)

print("\nData Types")
print(trips.dtypes)

print("\nMissing Values")
print(trips.isnull().sum())

print("\nInformation")
trips.info()

print("\nSummary Statistics")
print(trips.describe())

print(trips.columns)

First 5 Rows
  tpep_pickup_datetime tpep_dropoff_datetime  PULocationID  DOLocationID  \
0  2024-01-01 00:57:55   2024-01-01 01:17:43           186            79   
1  2024-01-01 00:03:00   2024-01-01 00:09:36           140           236   
2  2024-01-01 00:17:06   2024-01-01 00:35:01           236            79   
3  2024-01-01 00:36:38   2024-01-01 00:44:56            79           211   
4  2024-01-01 00:46:51   2024-01-01 00:52:57           211           148   

   passenger_count  trip_distance  fare_amount  total_amount  payment_type  
0              1.0           1.72         17.7         22.70             2  
1              1.0           1.80         10.0         18.75             1  
2              1.0           4.70         23.3         31.30             1  
3              1.0           1.40         10.0         17.00             1  
4              1.0           0.80          7.9         16.10             1  

Shape
(5972150, 9)

Data Types
tpep_pickup_datetime     datetime64[

In [4]:
trips["trip_minutes"] = (
    trips["tpep_dropoff_datetime"]
    - trips["tpep_pickup_datetime"]
).dt.total_seconds() / 60

In [5]:
trips = trips[
    (trips["total_amount"] > 0) &
    (trips["trip_distance"] > 0) &
    (trips["trip_minutes"] > 0)
]

print("Rows after cleaning:", trips.shape)

Rows after cleaning: (2871909, 10)


In [6]:
trips["pickup_month"] = trips["tpep_pickup_datetime"].dt.month

trips["pickup_date"] = trips["tpep_pickup_datetime"].dt.date

trips["pickup_hour"] = trips["tpep_pickup_datetime"].dt.hour

trips[[
    "pickup_month",
    "pickup_date",
    "pickup_hour",
    "trip_minutes"
]].head()

,pickup_month,pickup_date,pickup_hour,trip_minutes
0,1,2024-01-01,0,19.800000
1,1,2024-01-01,0,6.600000
2,1,2024-01-01,0,17.916667
3,1,2024-01-01,0,8.300000
4,1,2024-01-01,0,6.100000


In [7]:
pickup_zones = zones.rename(columns={
    "LocationID": "PULocationID",
    "Borough": "pickup_borough",
    "Zone": "pickup_zone"
})

trips = pd.merge(
    trips,
    pickup_zones[[
        "PULocationID",
        "pickup_borough",
        "pickup_zone"
    ]],
    on="PULocationID",
    how="left"
)

trips.head()

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,total_amount,payment_type,trip_minutes,pickup_month,pickup_date,pickup_hour,pickup_borough,pickup_zone
0,2024-01-01 00:57:55,2024-01-01 01:17:43,186,79,1.0,1.72,17.7,22.70,2,19.800000,1,2024-01-01,0,Manhattan,Penn Station/Madison Sq West
1,2024-01-01 00:03:00,2024-01-01 00:09:36,140,236,1.0,1.80,10.0,18.75,1,6.600000,1,2024-01-01,0,Manhattan,Lenox Hill East
2,2024-01-01 00:17:06,2024-01-01 00:35:01,236,79,1.0,4.70,23.3,31.30,1,17.916667,1,2024-01-01,0,Manhattan,Upper East Side North
3,2024-01-01 00:36:38,2024-01-01 00:44:56,79,211,1.0,1.40,10.0,17.00,1,8.300000,1,2024-01-01,0,Manhattan,East Village
4,2024-01-01 00:46:51,2024-01-01 00:52:57,211,148,1.0,0.80,7.9,16.10,1,6.100000,1,2024-01-01,0,Manhattan,SoHo


In [8]:
dropoff_zones = zones.rename(columns={
    "LocationID": "DOLocationID",
    "Borough": "dropoff_borough",
    "Zone": "dropoff_zone"
})

trips = pd.merge(
    trips,
    dropoff_zones[[
        "DOLocationID",
        "dropoff_borough",
        "dropoff_zone"
    ]],
    on="DOLocationID",
    how="left"
)

trips.head()

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,total_amount,payment_type,trip_minutes,pickup_month,pickup_date,pickup_hour,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone
0,2024-01-01 00:57:55,2024-01-01 01:17:43,186,79,1.0,1.72,17.7,22.70,2,19.800000,1,2024-01-01,0,Manhattan,Penn Station/Madison Sq West,Manhattan,East Village
1,2024-01-01 00:03:00,2024-01-01 00:09:36,140,236,1.0,1.80,10.0,18.75,1,6.600000,1,2024-01-01,0,Manhattan,Lenox Hill East,Manhattan,Upper East Side North
2,2024-01-01 00:17:06,2024-01-01 00:35:01,236,79,1.0,4.70,23.3,31.30,1,17.916667,1,2024-01-01,0,Manhattan,Upper East Side North,Manhattan,East Village
3,2024-01-01 00:36:38,2024-01-01 00:44:56,79,211,1.0,1.40,10.0,17.00,1,8.300000,1,2024-01-01,0,Manhattan,East Village,Manhattan,SoHo
4,2024-01-01 00:46:51,2024-01-01 00:52:57,211,148,1.0,0.80,7.9,16.10,1,6.100000,1,2024-01-01,0,Manhattan,SoHo,Manhattan,Lower East Side


In [9]:
unmatched_pickup = trips["pickup_borough"].isna().sum()

unmatched_dropoff = trips["dropoff_borough"].isna().sum()

print("Unmatched Pickup Zones:", unmatched_pickup)

print("Unmatched Dropoff Zones:", unmatched_dropoff)

print("Final Rows:", len(trips))

Unmatched Pickup Zones: 701
Unmatched Dropoff Zones: 10802
Final Rows: 2871909


In [10]:
pickup_zone_summary = trips.groupby(
    ["pickup_borough", "pickup_zone"],
    as_index=False
).agg(
    trip_count=("total_amount", "count"),
    total_revenue=("total_amount", "sum"),
    avg_revenue=("total_amount", "mean"),
    avg_distance=("trip_distance", "mean"),
    min_distance=("trip_distance", "min"),
    max_distance=("trip_distance", "max")
)

pickup_zone_summary = pickup_zone_summary.sort_values(
    by="total_revenue",
    ascending=False
)

pickup_zone_summary.head(10)

,pickup_borough,pickup_zone,trip_count,total_revenue,avg_revenue,avg_distance,min_distance,max_distance
202,Queens,JFK Airport,138480,11198083.05,80.864262,15.942619,0.01,10879.28
209,Queens,LaGuardia Airport,87708,5820947.01,66.367344,9.711433,0.01,176.43
143,Manhattan,Midtown Center,140244,3357585.44,23.941027,2.599400,0.01,38202.66
162,Manhattan,Upper East Side South,140171,2772175.99,19.777101,1.715744,0.01,971.80
156,Manhattan,Times Sq/Theatre District,103027,2766568.60,26.852850,2.971224,0.01,80.00
161,Manhattan,Upper East Side North,134037,2714234.44,20.249890,1.870805,0.01,58.81
149,Manhattan,Penn Station/Madison Sq West,102180,2462620.83,24.100811,2.303024,0.01,94.00
144,Manhattan,Midtown East,104388,2434506.48,23.321708,2.258858,0.01,71.18
135,Manhattan,Lincoln Square East,101854,2171677.63,21.321476,2.117787,0.01,58.78
145,Manhattan,Midtown North,83722,1971621.10,23.549618,2.540208,0.01,15400.32


In [11]:
payment_month_summary = trips.groupby(
    ["pickup_month", "payment_type"],
    as_index=False
).agg(
    trip_count=("total_amount", "count"),
    total_revenue=("total_amount", "sum"),
    avg_fare=("fare_amount", "mean"),
    min_fare=("fare_amount", "min"),
    max_fare=("fare_amount", "max")
)

payment_month_summary.head()

,pickup_month,payment_type,trip_count,total_revenue,avg_fare,min_fare,max_fare
0,1,0,117275,3060256.18,19.977361,-15.45,298.61
1,1,1,2298357,64527207.21,18.375810,0.00,650.00
2,1,2,422844,10089153.28,18.617193,0.00,2221.30
3,1,3,10613,235549.64,17.097690,0.00,700.00
4,1,4,22806,572064.61,19.728261,0.00,744.30


In [12]:
borough_hour_pivot = pd.pivot_table(
    trips,
    values="total_amount",
    index="pickup_hour",
    columns="pickup_borough",
    aggfunc="sum",
    fill_value=0
)

borough_hour_pivot.head()

pickup_borough,Bronx,Brooklyn,EWR,Manhattan,Queens,Staten Island,Unknown
pickup_hour,,,,,,,
0,1493.84,21632.10,0.0,1495182.93,621896.24,240.39,7208.49
1,1686.86,18782.71,0.0,1066337.43,214701.65,261.03,5078.79
2,516.15,15395.37,0.0,757983.79,81803.10,111.48,3514.76
3,1783.72,14569.60,0.0,533188.09,59278.42,126.82,2753.61
4,5868.80,19535.79,142.8,412417.78,55524.47,197.70,1847.75


In [13]:
borough_hour_long = borough_hour_pivot.reset_index().melt(
    id_vars="pickup_hour",
    var_name="pickup_borough",
    value_name="revenue"
)

borough_hour_long.head()

,pickup_hour,pickup_borough,revenue
0,0,Bronx,1493.84
1,1,Bronx,1686.86
2,2,Bronx,516.15
3,3,Bronx,1783.72
4,4,Bronx,5868.80


In [14]:
top_routes = trips.groupby(
    ["pickup_zone", "dropoff_zone"],
    as_index=False
).size()

top_routes = top_routes.rename(
    columns={"size": "trip_count"}
)

top_routes = top_routes.sort_values(
    by="trip_count",
    ascending=False
)

top_routes.head(10)

,pickup_zone,dropoff_zone,trip_count
22536,Upper East Side South,Upper East Side North,21647
22326,Upper East Side North,Upper East Side South,19207
22325,Upper East Side North,Upper East Side North,15200
22537,Upper East Side South,Upper East Side South,14116
14975,Midtown Center,Upper East Side South,10139
12993,Lincoln Square East,Upper West Side South,8829
22472,Upper East Side South,Midtown Center,8720
14974,Midtown Center,Upper East Side North,8667
22830,Upper West Side South,Lincoln Square East,8545
22905,Upper West Side South,Upper West Side North,8310


In [15]:
pickup_zone_summary.to_csv(
    "pickup_zone_summary.csv",
    index=False
)

borough_hour_pivot.to_csv(
    "borough_hour_pivot.csv"
)

borough_hour_long.to_csv(
    "borough_hour_long.csv",
    index=False
)

payment_month_summary.to_csv(
    "payment_month_summary.csv",
    index=False
)

print("All CSV files exported successfully")

All CSV files exported successfully


In [16]:
highest_revenue_zone = pickup_zone_summary.iloc[0]

print("Highest Revenue Pickup Zone")
print(highest_revenue_zone)

Highest Revenue Pickup Zone
pickup_borough         Queens
pickup_zone       JFK Airport
trip_count             138480
total_revenue     11198083.05
avg_revenue         80.864262
avg_distance        15.942619
min_distance             0.01
max_distance         10879.28
Name: 202, dtype: object


In [17]:
busiest_borough = trips.groupby(
    "pickup_borough"
).size().sort_values(
    ascending=False
)

print(busiest_borough.head(1))

pickup_borough
Manhattan    2575504
dtype: int64


In [18]:
manhattan_hour = borough_hour_pivot["Manhattan"].idxmax()

manhattan_revenue = borough_hour_pivot["Manhattan"].max()

print("Highest Revenue Hour in Manhattan:", manhattan_hour)

print("Revenue:", manhattan_revenue)

Highest Revenue Hour in Manhattan: 18
Revenue: 4334322.32
